# Phase 6 — which network, measured rather than assumed

Four training runs that differ in the network and in nothing else. The baseline
is a U-Net with a ResNet-34 encoder because that is where the project started,
not because anything measured said so, and this notebook is what replaces the
assumption with a number.

| run | decoder | encoder | what it isolates |
|---|---|---|---|
| a | Unet | resnet34 | the reference, at this epoch count |
| b | Unet | tu-convnext_tiny | the encoder alone |
| c | UPerNet | tu-convnext_tiny | the decoder, on b's encoder |
| d | Segformer | mit_b2 | a different family |

Run *d* trains on batches of two where the others use four. A transformer
compares every patch with every other, so at 1024 pixels a batch of four does
not fit a T4 and this run was lost to that on the first attempt. Its number is
therefore not on quite the same footing as the other three, and saying so is
part of quoting it.

Why these and not others. Filaments are thin, they cover 0.35% of a frame, and
the ones missing from the probability map entirely are the small ones: a
correct box drawn around 266 of fold 0's filaments contains no pixel above 0.05.
An encoder-decoder that halves its resolution five times has the most to lose
there. UPerNet pools context over the whole frame before decoding; the
Segformer encoder attends globally from its first stage. Both are reasons about
this data, not about publication dates.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2**
2. Internet: **on**
3. Add the competition data as an input
4. **Save Version** with *Save output* on, or the checkpoints and maps are discarded

Expected wall clock on one T4: roughly 50 minutes for run a, 60 to 90 for each
of the others, so four to five hours in total, plus twenty minutes writing
probability maps.

**Re-running after one of them failed.** A run whose `best.pt` is already in
`/kaggle/working` is scored from that file instead of being trained again, so
recovering a single crashed run costs only that run. Starting from a fresh
session, attach the earlier version's output as an input and copy the finished
directories into `/kaggle/working` first; otherwise every run trains again.

## 1. Clone the repository and put it on the path

The repository is cloned rather than installed, because `configs/paths.yaml`
and the frozen splits in `configs/splits/` sit beside the package rather than
inside it. Set `REF` to the branch or commit this run is to be reproducible
from.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase6-architecture"  # branch or commit hash
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())
if torch.cuda.is_available():
    print("name:", torch.cuda.get_device_name(0))

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

dataset_root = candidates[0].parent.parent
os.environ["MAGFILO_ROOT"] = str(dataset_root)
print("MAGFILO_ROOT =", dataset_root)

paths = load_paths().require_dataset()
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))

## 3. The four runs

The ground truth is encoded once and handed to every scoring call: it takes as
long as the inference and does not depend on the model.

Scoring uses the post-processing Phase 3 settled on — threshold 0.5, minimum
area 400, rejoining within 24 pixels — for all four runs. The defaults in the
code are not that configuration, so they are named explicitly here. Two runs
are comparable only when the chain behind them is the same.

In [ ]:
from dataclasses import replace

from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.training.config import TrainConfig

RUNS = ["a_unet_resnet34", "b_unet_convnext", "c_upernet_convnext", "d_segformer_mit"]

# The configuration Phase 3 settled on, applied identically to every run.
SCORING = {"threshold": 0.5, "min_area": 400, "join_gap": 24.0}

dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(0, f"{CHECKOUT}/configs/splits").val
print(f"fold 0: {len(val_stems)} validation frames")

configs = {}
for name in RUNS:
    configs[name] = replace(
        TrainConfig.from_yaml(f"{CHECKOUT}/configs/phase6/{name}.yaml"),
        num_workers=2,
        output_dir=Path(f"/kaggle/working/{name}"),
    )
    config = configs[name]
    print(f"{name:<22} {config.architecture:<10} {config.encoder:<18} {config.epochs} epochs")

In [ ]:
from filament.submit.rle import masks_to_gt_df, read_submission, write_submission

gt_path = Path("/kaggle/working/gt_fold0.csv")
if gt_path.exists():
    gt_df = read_submission(gt_path)
else:
    gt_df = masks_to_gt_df(dataset, val_stems)
    write_submission(gt_df, gt_path)
print(f"{len(gt_df)} ground-truth filaments")

In [ ]:
import json
import logging
import time

from filament.evaluation import evaluate
from filament.training.loop import load_checkpoint, train

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

results = {}
for name in RUNS:
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    checkpoint = configs[name].output_dir / "best.pt"

    # Already trained: a run that crashed part-way through the four should not
    # cost the three that succeeded. Delete the directory to force a retrain.
    if checkpoint.exists():
        print(f"{name}: reusing the checkpoint already in {checkpoint.parent}")
        training_minutes = None
        best_epoch = torch.load(checkpoint, map_location="cpu", weights_only=False)["epoch"]
        best_val_loss = None
    else:
        started = time.perf_counter()
        outcome = train(configs[name])
        training_minutes = round((time.perf_counter() - started) / 60, 1)
        best_epoch = outcome.best_epoch
        best_val_loss = round(outcome.best_val_loss, 4)
        checkpoint = outcome.checkpoint

    model, _ = load_checkpoint(checkpoint)
    evaluation, _ = evaluate(
        model,
        dataset,
        paths.train_images,
        val_stems,
        size=configs[name].image_size,
        device="cuda",
        gt_df=gt_df,
        **SCORING,
    )
    results[name] = evaluation.to_dict() | {
        "architecture": configs[name].architecture,
        "encoder": configs[name].encoder,
        "batch_size": configs[name].batch_size,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "training_minutes": training_minutes,
    }
    Path(f"/kaggle/working/{name}/eval_fold0.json").write_text(json.dumps(results[name], indent=2))
    print(evaluation, f"| {training_minutes} min")

    # Freeing it matters: four networks held at once will not fit alongside the
    # next run's activations.
    del model
    torch.cuda.empty_cache()

Path("/kaggle/working/phase6_summary.json").write_text(json.dumps(results, indent=2))

## 4. The comparison

The incumbent is **PQ 0.3756**, which is this same network trained for fifteen
epochs and scored through the post-processing Phase 3 tuned. Run *a* is that
network at twenty, so the gap between the two is the epoch count alone and says
how much of any difference below is length rather than architecture.

Adoption needs **+0.01 over run a** and an explanation of why it moved. SQ and
RQ are shown apart because they say different things: SQ is how well a matched
filament is drawn, RQ is how many were matched at all. Nothing tried so far has
moved SQ out of 0.645 to 0.665, and whether any of these does is the question
this notebook exists to answer.

In [ ]:
import pandas as pd

table = pd.DataFrame(results).T[
    [
        "architecture",
        "encoder",
        "pq",
        "sq",
        "rq",
        "tp",
        "fp",
        "fn",
        "fused",
        "split",
        "best_epoch",
        "training_minutes",
    ]
]
table["pq_vs_a"] = (table["pq"] - table.loc["a_unet_resnet34", "pq"]).round(4)
table

## 5. Probability maps, for every run

Saved for all of them rather than for the winner alone. The threshold, the
minimum area and the rejoining distance used above were tuned on the incumbent
U-Net, and scoring a different network through them cannot tell a worse network
from one whose probabilities simply sit elsewhere. The two failures seen so far
point opposite ways -- one network emitting a hundred predictions too many and
another ninety true ones too few -- which is what a mismatched threshold looks
like as much as it is what a worse network looks like. Settling that needs each
network's own maps.

The sweeps then run on the maps rather than on the model, on a CPU, so no
further GPU time goes into tuning whatever wins.

Float16 keeps one run's fold to about 300 MB, so four come to 1.2 GB.

In [ ]:
import numpy as np

from filament.data.image import load_grayscale
from filament.evaluation import predict_probability

for name in RUNS:
    checkpoint = Path(f"/kaggle/working/{name}/best.pt")
    if not checkpoint.exists():
        print(f"{name}: no checkpoint, skipped")
        continue

    model, _ = load_checkpoint(checkpoint)
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
    maps_dir.mkdir(parents=True, exist_ok=True)
    for stem in val_stems:
        probability = predict_probability(
            model,
            load_grayscale(paths.train_images / f"{stem}.jpeg"),
            size=configs[name].image_size,
            device="cuda",
        )
        np.save(maps_dir / f"{stem}.npy", probability.astype(np.float16))

    written = sorted(maps_dir.glob("*.npy"))
    megabytes = sum(item.stat().st_size for item in written) / 1e6
    print(f"{name}: {len(written)} maps, {megabytes:.0f} MB")

    del model
    torch.cuda.empty_cache()

## 6. Each network at its own post-processing

Section 3 scored every run through one chain, which is what makes the networks
comparable. It is not what decides whether a network is any good: the
threshold, the minimum area and the rejoining distance were tuned on the
incumbent U-Net, and a network whose probabilities sit elsewhere is judged by
a gate that was not built for it. The two failures seen so far point opposite
ways, one emitting a hundred predictions too many and another ninety true ones
too few, which is what a mismatched threshold looks like as much as it is what
a worse network looks like.

So both readings are kept, as Phase 5 kept them: at the shared setting, and at
each network's own best. **The adoption decision is made on the shared
setting.** The retuned figure says whether a rejected network was rejected for
the right reason.

Nothing here needs a GPU. It reads the maps written above.

**One check before the numbers are used.** The sweep and the scoring in section
3 are two different code paths, and they do not treat the edge of the solar
disk identically: scoring shrinks the 16-pixel margin to match the half-size
map, while the sweep does not. A prediction sitting just past the limb can
therefore survive one path and not the other. If the two disagree at the same
setting, a difference between them measures the path rather than the setting,
and the retuned figures are not usable until that is settled. The cell prints
both so the question is answered rather than assumed.

In [ ]:
from filament.data.disk import detect_disk
from filament.postprocess.search import grid, load_maps, sweep

# The saved maps are the raw model output rather than a disk-masked one, so the
# sweep has to be given the disk. It is the same disk for every configuration,
# so it is found once and reused.
disks = {
    stem: detect_disk(load_grayscale(paths.train_images / f"{stem}.jpeg")).scaled(0.5)
    for stem in val_stems
}

# Threshold is the axis worth the most here. Phase 3 found it inert on the
# incumbent -- every value from 0.3 to 0.7 within 0.001 -- but that is a
# property of one network's probabilities, not of the problem.
SETTINGS = grid(
    threshold=[0.3, 0.4, 0.5, 0.6, 0.7],
    min_area=[200, 400, 600],
    join_gap=[0.0, 24.0],
)
print(f"{len(SETTINGS)} settings per run")

retuned = {}
for name in RUNS:
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
    if not maps_dir.exists():
        print(f"{name}: no maps, skipped")
        continue

    maps = load_maps(maps_dir, val_stems)
    outcome = sweep(maps, gt_df, SETTINGS, disks=disks)
    outcome.table().to_csv(f"/kaggle/working/sweep_{name}.csv", index=False)

    shared = [point for point in outcome.points if point.setting.values == SCORING]
    if shared and name in results:
        print(
            f"{name}: same setting -- sweep {shared[0].pq.pq:.4f}, "
            f"section 3 {results[name]['pq']:.4f}"
        )

    best = outcome.best
    retuned[name] = {
        "setting": dict(best.setting.values),
        "pq": round(best.pq.pq, 4),
        "sq": round(best.pq.sq, 4),
        "rq": round(best.pq.rq, 4),
        "tp": best.pq.tp,
        "fp": best.pq.fp,
        "fn": best.pq.fn,
    }
    print(f"{name}: best {best.pq.pq:.4f} at {best.setting}")
    del maps

Path("/kaggle/working/phase6_retuned.json").write_text(json.dumps(retuned, indent=2))

## 7. What to record

Into the lab notebook, for every one of the four runs and not only the winner —
the ones that did not work are what the ablation table in the final report is
made of:

- PQ, SQ, RQ, TP, FP, FN, and the fused and split counts
- best epoch and training time
- the commit hash this notebook cloned

Do not submit from here. The comparison is decided on fold 0 locally; folds 1
to 4 stay untouched until a configuration is frozen.